# Audited spinal morphology analysis
Four nominal categories. This notebook uses the corrected repository scripts; it does not train a model or manufacture probabilities. Start from the corrected repository, not historical v1.0.0.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'scripts/export_metrics.py').is_file():
    if (ROOT.parent / 'scripts/export_metrics.py').is_file():
        ROOT = ROOT.parent
    elif (ROOT / 'axspa-yolo-cls/scripts/export_metrics.py').is_file():
        ROOT = ROOT / 'axspa-yolo-cls'
    else:
        raise RuntimeError('Upload/extract the corrected repository and set ROOT to its folder. Do not use historical v1.0.0 scripts.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)
OUT = ROOT/'results'
OUT.mkdir(exist_ok=True)


## Published hard-label matrices
These support label-based metrics only. Figure 1 is transposed to true rows/predicted columns. Accuracy uses exact intervals; label bootstrap uses reconstructed pair order.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/export_metrics.py'),'--published-matrices','--output',str(OUT/'confusion_matrix_audit.json')],check=True)


## Original probability arrays
Supply authorized internal/external NPZ paths. The fixed external threshold was derived internally. The Youden optimum calculated on any input must not be treated as externally validated when derived on that same input.

In [ ]:
INTERNAL = ROOT/'data/internal_probs.npz'
EXTERNAL = ROOT/'data/external_probs.npz'
for label,path in [('internal',INTERNAL),('external',EXTERNAL)]:
    if not path.is_file(): raise FileNotFoundError(path)
    subprocess.run([sys.executable,str(ROOT/'scripts/export_metrics.py'),'--input',str(path),'--positive-indices','2','3','--threshold','0.584019549792572','--output',str(OUT/(label+'_probability_audit.json'))],check=True)


## Results
This does not independently authenticate historical checkpoint inference. Raw images and weights are required for a separate new inference run using scripts/evaluate_folder.py.

In [ ]:
import json
print(json.loads((OUT/'external_probability_audit.json').read_text())['overall'])

## Reproduce probability figures
ROC, calibration and binary ROC are regenerated with bin counts and SHA-256 provenance. Eigen-CAM requires the original model and images; no heatmap reproduction is claimed here.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/regenerate_probability_figures.py'),'--inputs',str(ROOT/'data'),'--output',str(OUT/'figures')],check=True)